<a href="https://colab.research.google.com/github/KahPrisco/Bootcamp/blob/main/CTE's_e_Subconsultas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.9/33.9 MB 14.2 MB/s eta 0:00:00


In [ ]:
import mysql.connector
import pandas as pd

In [ ]:
!curl ipecho.net/plain

34.31.213.243

In [ ]:
host = '35.222.106.159' # endereço do banco externo que eu quero acessar
user = 'root'
password = ''
database = 'locadora'

In [ ]:
def consulta(query, tabela):
  connection = mysql.connector.connect(
      host = host,
      user = user,
      password = password,
      database = database
  )
  cursor = connection.cursor()

  try:
    cursor.execute(query)
    result = cursor.fetchall()
    globals()[tabela] = pd.DataFrame(result, columns=cursor.column_names)
    display(globals()[tabela])
  finally:
    cursor.close()
    connection.close()

In [ ]:
consulta('''
SHOW databases;
''',''' ''')

,Database
0,faculdade
1,information_schema
2,locadora
3,mysql
4,performance_schema
5,rh
6,sys


In [ ]:
consulta('''
SELECT * FROM cliente
''',''' ''')

,codcliente,nome,cidade,sexo,estado,estadocivil
0,1,Ana Silva,Duque de Caxias,F,RJ,C
1,2,Bruna Pereira,Niterói,F,RJ,C
2,3,Túlio Nascimento,Duque de Caxias,M,RJ,S
3,4,Fernando Souza,Campinas,M,SP,S
4,5,Lúcia Andrade,São Paulo,F,SP,C


In [ ]:
'''
1. Lista de cidades com clientes que já alugaram carros
Liste todas as cidades em que há clientes que realizaram aluguéis.
Dica: Use uma subconsulta para verificar clientes na tabela aluguel.
'''

consulta('''
SELECT DISTINCT cliente.nome, cliente.cidade
FROM cliente
JOIN aluguel
ON cliente.codcliente = aluguel.codcliente
WHERE cliente.codcliente IN (
    SELECT al.codcliente
    FROM aluguel AS al
);


''',tabela='')

,nome,cidade
0,Ana Silva,Duque de Caxias
1,Bruna Pereira,Niterói
2,Túlio Nascimento,Duque de Caxias
3,Lúcia Andrade,São Paulo


In [ ]:
'''
2. Modelos de carros mais caros que a média Liste os modelos de carros cujo valor seja maior que o valor médio de todos os carros.
'''
consulta('''
WITH mais_caro_media AS (
  SELECT modelo, valor
  FROM carro
  WHERE valor > (SELECT AVG(valor) FROM carro)
)
SELECT * FROM mais_caro_media;
''', tabela='')


,modelo,valor
0,Argo,150.0
1,Onix,170.0
2,Polo,150.0


In [ ]:
consulta('''
SELECT AVG(valor)
FROM carro;
''', tabela='')

,AVG(valor)
0,138.0


In [ ]:
'''
3 - Clientes que alugaram carros da marca Ford Liste os nomes dos clientes que alugaram pelo menos um carro da marca Ford.
Dica: Use uma subconsulta para filtrar a marca.
'''
consulta('''SELECT DISTINCT  cli.nome, mar.marca
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
JOIN marca AS mar
ON mar.codmarca = car.codmarca
WHERE mar.codmarca = (SELECT codmarca FROM marca WHERE marca = 'Ford');

''','')

,nome,marca
0,Bruna Pereira,Ford
1,Ana Silva,Ford
2,Túlio Nascimento,Ford


In [ ]:
'''
4 - Carros alugados apenas por clientes de São Paulo Liste os modelos de carros que foram alugados exclusivamente por clientes do estado de São Paulo.
Dica: Use NOT IN com uma subconsulta.
'''

consulta('''SELECT DISTINCT cli.nome, car.modelo , cli.estado
FROM carro AS car
JOIN aluguel AS al
ON car.codcarro = al.codcarro
JOIN cliente AS cli
ON al.codcliente = cli.codcliente
WHERE cli.estado = 'SP' AND car.codcarro NOT IN
(SELECT DISTINCT al.codcarro
FROM aluguel as al
JOIN cliente as cli
ON al.codcliente = cli.codcliente
WHERE cli.estado <> 'SP');

''','')




,nome,modelo,estado


In [ ]:
'''
5 - Datas mais movimentadas Liste as datas em que houve mais de 2 aluguéis.
Dica: Use uma subconsulta com GROUP BY e HAVING.
'''
consulta('''
SELECT data_aluguel, COUNT(codcarro) AS total_alugueis
FROM aluguel
GROUP BY data_aluguel
HAVING COUNT(codcarro) > 2;

''','')



,data_aluguel,total_alugueis


In [ ]:
'''
6 - Clientes que nunca alugaram carros da marca Fiat Liste os nomes dos clientes que nunca alugaram carros da marca Fiat.
'''
consulta('''
SELECT DISTINCT cli.nome
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
JOIN marca AS mar
ON mar.codmarca = car.codmarca
WHERE cli.codcliente NOT IN (
  SELECT al.codcliente
  FROM aluguel AS al
  JOIN carro AS car
  ON al.codcarro = car.codcarro
  JOIN marca AS mar
  ON car.codmarca = mar.codmarca
  WHERE mar.marca = 'Fiat'
)
''','')


,nome
0,Bruna Pereira
1,Ana Silva


In [ ]:
'''
7 - Carros mais alugados em cidades específicas Liste os modelos de carros que foram alugados em cidades onde residem mais de 3 clientes.
Dica: Use uma subconsulta para encontrar essas cidades.
'''
consulta('''
SELECT DISTINCT cli.nome, car.modelo, cli.cidade
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
WHERE cli.cidade IN (
  SELECT cidade
  FROM cliente
  GROUP BY cidade
  HAVING COUNT(codcliente) >3
);
''','')

,nome,modelo,cidade


In [ ]:
'''
8 - Valor total de aluguéis por cliente com subconsulta Liste os nomes dos clientes e o valor total gasto em aluguéis.
Dica: Use uma subconsulta para calcular a soma por cliente.
'''
consulta('''
SELECT cli.nome, SUM(car.valor) AS total
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
GROUP BY nome

''','')


,nome,total
0,Túlio Nascimento,250.0
1,Bruna Pereira,370.0
2,Ana Silva,400.0
3,Lúcia Andrade,300.0


In [ ]:
consulta('''
SELECT c.nome,
       (SELECT SUM(ca.valor)
        FROM aluguel a
        JOIN carro ca ON a.codcarro = ca.codcarro
        WHERE a.codcliente = c.codcliente) AS total_gasto
FROM cliente c;
''','')

,nome,total_gasto
0,Ana Silva,400.0
1,Bruna Pereira,370.0
2,Túlio Nascimento,250.0
3,Fernando Souza,NaN
4,Lúcia Andrade,300.0


In [ ]:
'''
9 - Clientes que gastaram acima da média Liste os nomes dos clientes cujo valor total gasto com aluguéis está acima da média geral.
'''
consulta('''
WITH total_gasto_cliente AS (
  SELECT cli.nome, SUM(car.valor) AS total_gasto_alugueis
  FROM cliente AS cli
  JOIN aluguel AS al ON cli.codcliente = al.codcliente
  JOIN carro AS car ON al.codcarro = car.codcarro
  GROUP BY cli.nome
)

SELECT nome, total_gasto_alugueis
FROM total_gasto_cliente
WHERE total_gasto_alugueis > (
  SELECT AVG(total_gasto_alugueis) FROM total_gasto_cliente
);
''', '')



,nome,total_gasto_alugueis
0,Bruna Pereira,370.0
1,Ana Silva,400.0


In [ ]:
'''
10 - Clientes que mais alugaram em um único dia Liste os nomes dos clientes que realizaram o maior número de aluguéis em um único dia.
Dica: Use MAX em uma subconsulta.
'''
consulta('''
SELECT cli.nome, al.data_aluguel, COUNT(al.codcliente) AS numero_aluguel
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
GROUP BY data_aluguel, cli.nome

HAVING numero_aluguel = (
  SELECT MAX(alugueis_por_dia)
  FROM (
    SELECT COUNT(al.codcliente) AS alugueis_por_dia
    FROM aluguel
    GROUP BY codcliente, data_aluguel
  ) AS sub
);

''','''''')

,nome,data_aluguel,numero_aluguel
0,Túlio Nascimento,2023-04-01,1
1,Bruna Pereira,2023-04-02,1
2,Bruna Pereira,2023-04-03,1
3,Bruna Pereira,2023-04-04,1
4,Ana Silva,2023-04-05,1
5,Ana Silva,2023-04-13,1
6,Ana Silva,2023-04-15,1
7,Lúcia Andrade,2023-04-19,1
8,Lúcia Andrade,2023-04-21,1
9,Túlio Nascimento,2023-04-25,1


In [ ]:
'''
11 - Clientes que alugaram todos os carros de uma marca Liste os nomes dos clientes que alugaram todos os carros disponíveis da marca Chevrolet.
Dica: Compare a contagem de carros alugados por cliente com a contagem total de carros da marca.
'''
consulta('''
SELECT cli.nome
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
JOIN marca AS mar
ON car.codmarca = mar.codmarca
WHERE mar.marca = 'Chevrolet'
GROUP BY cli.codcliente, cli.nome;

HAVING COUNT(DISTINCT car.codcarro) = (
    SELECT COUNT(car.codcarro)
    FROM carro AS car
    WHERE mar.marca = 'Chevrolet'
);
''','''''')



,nome
0,Bruna Pereira


In [ ]:
'''
12 - Carros alugados mais de uma vez no mesmo dia Liste os modelos de carros que foram alugados mais de uma vez na mesma data.
'''
consulta('''
SELECT DISTINCT al.data_aluguel, car.modelo, COUNT(car.modelo)
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
GROUP BY data_aluguel, car.modelo
HAVING COUNT(car.modelo) > 1;
''','''''')

,data_aluguel,modelo,COUNT(car.modelo)


In [ ]:
'''
13 - Carros mais caros alugados por clientes de um estado Liste os modelos dos carros mais caros alugados por clientes do estado do Rio de Janeiro.
'''
consulta('''
SELECT cli.nome, cli.estado, car.modelo, car.valor
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
WHERE cli.estado = 'RJ' AND car.valor = (SELECT MAX(valor) FROM carro)
GROUP BY cli.nome, cli.estado, car.modelo, car.valor;
''','''''')

,nome,estado,modelo,valor
0,Bruna Pereira,RJ,Onix,170.0


In [ ]:
'''
14 - Datas de aluguel de carros populares Liste as datas em que foram alugados carros das marcas Ford ou Volkswagen.
'''
consulta('''
SELECT al.data_aluguel, mar.marca
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
JOIN marca AS mar
ON mar.codmarca = car.codmarca
WHERE mar.marca = 'Ford' OR mar.marca = 'Volkswagen';
''','''''')

,data_aluguel,marca
0,2023-04-02,Ford
1,2023-04-03,Ford
2,2023-04-15,Ford
3,2023-04-25,Ford
4,2023-04-05,Volkswagen
5,2023-04-13,Volkswagen


In [ ]:
consulta('''
SELECT al.data_aluguel, mar.marca
FROM cliente AS cli
JOIN aluguel AS al
ON cli.codcliente = al.codcliente
JOIN carro AS car
ON al.codcarro = car.codcarro
JOIN marca AS mar
ON mar.codmarca = car.codmarca
WHERE car.codmarca IN (
    SELECT codmarca FROM marca
    WHERE marca = 'Ford' OR marca = 'Volkswagen'
);
''','')


,data_aluguel,marca
0,2023-04-02,Ford
1,2023-04-03,Ford
2,2023-04-15,Ford
3,2023-04-25,Ford
4,2023-04-05,Volkswagen
5,2023-04-13,Volkswagen


In [ ]:
'''
15 - Carros nunca alugados Liste os modelos de carros que nunca foram alugados.
'''
consulta('''
SELECT car.modelo, mar.marca
FROM carro AS car
JOIN marca AS mar
ON car.codmarca = mar.codmarca
WHERE car.codcarro NOT IN (
  SELECT codcarro FROM aluguel
);
''','')

,modelo,marca
0,Kwid,Renault


In [ ]:
'''
16 - Total de aluguéis por cliente
Crie uma CTE para calcular o número total de aluguéis realizados por cada cliente e liste os clientes que alugaram mais de 2 vezes.
'''
consulta('''
WITH total_alugueis_cliente AS (
    SELECT cli.nome, COUNT(al.codaluguel) AS total
    FROM cliente AS cli
    JOIN aluguel AS al
    ON cli.codcliente = al.codcliente
    GROUP BY cli.nome
),
alugaram_mais_duas AS (
    SELECT nome, total
    FROM total_alugueis_cliente
    WHERE total > 2
)
SELECT * FROM alugaram_mais_duas;
''','')


,nome,total
0,Ana Silva,3
1,Bruna Pereira,3


In [ ]:
'''
17 - Total gasto por cliente Crie uma CTE para calcular o valor total gasto por cliente e liste apenas os clientes que gastaram mais de R$300,00.
'''

consulta('''
WITH total_gasto_cliente AS (
    SELECT cli.nome, SUM(car.valor) AS total
    FROM cliente AS cli
    JOIN aluguel AS al
    ON cli.codcliente = al.codcliente
    JOIN carro AS car
    ON car.codcarro = al.codcarro
    GROUP BY cli.nome
),
alugaram_mais_300 AS (
    SELECT nome, total
    FROM total_gasto_cliente
    WHERE total > 300
)
SELECT * FROM alugaram_mais_300;
''','')

,nome,total
0,Bruna Pereira,370.0
1,Ana Silva,400.0


In [ ]:
'''
18 - Carros mais alugados Crie uma CTE para calcular o número de vezes que cada modelo de carro foi alugado e liste apenas os modelos alugados mais de 3 vezes.
'''

consulta('''
WITH modelo_mais_alugado AS (
    SELECT car.modelo, COUNT(car.codcarro) AS total
    FROM aluguel AS al
    JOIN carro AS car
    ON al.codcarro = car.codcarro
    GROUP BY car.modelo
),

alugaram_mais_3 AS (
    SELECT modelo, total
    FROM modelo_mais_alugado
    WHERE total > 3
)

SELECT * FROM alugaram_mais_3;

''','')

,modelo,total
0,Ka,4


In [ ]:
'''
19 - Estados com maior número de clientes Crie uma CTE para calcular o número de clientes por estado e liste os estados com mais de 3 clientes.
'''
consulta('''
WITH estado_mais_cliente AS (
    SELECT cli.estado, COUNT(cli.nome) AS total
    FROM cliente AS cli
    GROUP BY cli.estado
),

estado_mais_3 AS (
    SELECT estado, total
    FROM estado_mais_cliente
    WHERE total > 3
)

SELECT * FROM estado_mais_3;

''','')

,estado,total


In [ ]:
'''
20 - Clientes com aluguéis consecutivos Crie uma CTE para listar os clientes que realizaram aluguéis em dias consecutivos.
Dica: Use a função DATEDIFF.
'''

consulta('''
WITH cliente_alugueis AS (
    SELECT cli.codcliente, cli.nome, al.data_aluguel
    FROM cliente AS cli
    JOIN aluguel AS al
    ON cli.codcliente = al.codcliente
),

cliente_alugueis_consecutivos AS (
    SELECT a1.nome, a1.data_aluguel AS data1, a2.data_aluguel AS data2
    FROM cliente_alugueis a1
    JOIN cliente_alugueis a2
    ON a1.codcliente = a2.codcliente
    WHERE DATEDIFF(a2.data_aluguel, a1.data_aluguel) = 1
)

SELECT * FROM cliente_alugueis_consecutivos;
''','')



,nome,data1,data2
0,Bruna Pereira,2023-04-02,2023-04-03
1,Bruna Pereira,2023-04-03,2023-04-04
